# SniffTest Train Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skrript/sniff-test/blob/main/snifftest_train.ipynb)

Self-contained GRPO training notebook for SniffTest. Uses Unsloth's standard `reward_funcs` API (no custom `rollout_func`). The model generates a **full action sequence** in one shot; reward functions evaluate that sequence against the environment.


## 1. Install

In [ ]:
%%capture
import os, sys
# Install Unsloth + training stack
!pip install -qqq "unsloth[colab-new]" "trl>=0.17.0" "peft>=0.19.1" "accelerate>=1.13.0" "datasets>=3.6.0" "bitsandbytes>=0.45.0" matplotlib python-dotenv


## 2. Clone repo & install package

In [ ]:
import os, sys, subprocess, socket, time, signal, atexit
from pathlib import Path

REPO_URL = os.getenv("SNIFFTEST_REPO_URL", "https://github.com/skrript/sniff-test.git")
WORKSPACE = Path.cwd().resolve()
REPO_DIR = (WORKSPACE if (WORKSPACE/"pyproject.toml").exists() else WORKSPACE/"sniff-test")
BASE_URL  = os.getenv("SNIFFTEST_BASE_URL", "http://127.0.0.1:8000")
MODEL_NAME = os.getenv("SNIFFTEST_MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
ARTIFACT_DIR = (REPO_DIR / "temp/snifftest_train")

if not REPO_DIR.exists():
    subprocess.run(["git","clone",REPO_URL,str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","."], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo:", REPO_DIR)
print("Model:", MODEL_NAME)


## 3. HuggingFace login

In [ ]:
import os
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if not HF_TOKEN:
    from huggingface_hub import notebook_login
    notebook_login()
    HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if HF_TOKEN:
    os.environ.setdefault("HF_TOKEN", HF_TOKEN)
    os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", HF_TOKEN)
print("HF token configured:", bool(HF_TOKEN))


## 4. Load model (Unsloth)

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = int(os.getenv("SNIFFTEST_MAX_SEQ_LENGTH", "2048"))
LORA_RANK      = int(os.getenv("SNIFFTEST_LORA_RANK", "16"))
LOAD_IN_4BIT   = os.getenv("SNIFFTEST_LOAD_IN_4BIT","1").lower() in {"1","true","yes"}

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    load_in_4bit  = LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r                        = LORA_RANK,
    target_modules           = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha               = LORA_RANK * 2,
    lora_dropout             = 0,
    bias                     = "none",
    use_gradient_checkpointing= "unsloth",
    random_state             = 42,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Model loaded:", MODEL_NAME)


## 5. Start environment server

In [ ]:
SERVER_PROCESS = None

def _port(url):
    from urllib.parse import urlparse
    p = urlparse(url).port
    return p or (443 if url.startswith("https") else 80)

SERVER_PORT = _port(BASE_URL)

def start_server():
    global SERVER_PROCESS
    if SERVER_PROCESS and SERVER_PROCESS.poll() is None:
        return
    env = {**os.environ, "PYTHONPATH": str(REPO_DIR)}
    SERVER_PROCESS = subprocess.Popen(
        [sys.executable,"-m","uvicorn","server.app:app","--host","127.0.0.1","--port",str(SERVER_PORT)],
        cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    deadline = time.time() + 30
    while time.time() < deadline:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", SERVER_PORT)) == 0:
                break
        time.sleep(0.25)
    print(f"Server up at {BASE_URL} (pid {SERVER_PROCESS.pid})")

def stop_server():
    global SERVER_PROCESS
    if SERVER_PROCESS and SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.send_signal(signal.SIGTERM)
    SERVER_PROCESS = None

atexit.register(stop_server)
start_server()


## 6. Smoke test

In [ ]:
async def _smoke():
    from client import SniffTestEnv
    async with SniffTestEnv(base_url=BASE_URL) as env:
        r = await env.reset(task_level="easy")
        print("Claim:", r.observation.claim[:80])
        print("Sources:", len(r.observation.available_sources))

await _smoke()


## 7. Reward functions

**Key fix vs old notebook:** reward functions run inside themselves against `SniffTestEnvironment` directly. No `rollout_func`. The model generates a **full JSON action array** in one shot; we parse and evaluate it.

This matches Unsloth's expected API — `GRPOTrainer` calls each reward function with `completions` (list of message-dicts) and routes them correctly.

In [ ]:
import json, textwrap
from typing import Any

try:
    from server.snifftest_environment import SniffTestEnvironment
    from models import InvestigateAction
except ImportError:
    from snifftest_env.server.snifftest_environment import SniffTestEnvironment
    from snifftest_env.models import InvestigateAction

SYSTEM_PROMPT = textwrap.dedent("""
    You are an expert fact-checker investigating claims for accuracy.

    Available actions (return as JSON array, ALL actions in one response):
    [{"action_type": "search", "query": "your search query"},
     {"action_type": "open_source", "source_id": "src_xxx"},
     {"action_type": "cross_reference", "source_ids": ["src_xxx", "src_yyy"]},
     {"action_type": "trace_origin", "source_id": "src_xxx"},
     {"action_type": "check_metadata", "source_id": "src_xxx"},
     {"action_type": "submit_verdict", "verdict": "true|false|misleading|unverifiable",
      "confidence": 0.0, "justification": "reasoning citing source IDs"}]

    Strategy: start with search, open key sources, end with submit_verdict.
    Return ONLY the JSON array. No other text.
""").strip()

VALID_VERDICTS = {"true","false","misleading","unverifiable"}
MAX_ACTIONS    = 10
_PRINTER = 0

def _extract_actions(text: str):
    """Extract first JSON array from model output."""
    text = text.strip()
    for start in [i for i,c in enumerate(text) if c=="["]:
        try:
            parsed, _ = json.JSONDecoder().raw_decode(text[start:])
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            continue
    # fallback: try whole text
    try:
        return json.loads(text)
    except Exception:
        return None

def _run_episode(actions_raw, task_level="easy"):
    """Run a full episode with parsed actions; return grade dict."""
    env = SniffTestEnvironment(enable_adversarial=False)
    obs = env.reset(task_level=task_level)
    for raw in (actions_raw or [])[:MAX_ACTIONS]:
        if getattr(obs, "done", False):
            break
        try:
            action = InvestigateAction(**raw)
            obs = env.step(action)
        except Exception:
            break
    return getattr(env.state, "grade_result", {}) or {}


def _get_text(completion):
    """Extract assistant text from a completion (list of message dicts or string)."""
    if isinstance(completion, list):
        for m in reversed(completion):
            if isinstance(m, dict) and m.get("role") == "assistant":
                return m.get("content", "")
        # fallback: last item's content
        if completion and isinstance(completion[-1], dict):
            return completion[-1].get("content", "")
    return str(completion)


def _task_level_from_prompt(prompts, idx):
    """Extract task_level from the prompt dataset row."""
    if prompts and idx < len(prompts):
        p = prompts[idx]
        if isinstance(p, list):
            for m in p:
                if isinstance(m, dict):
                    c = m.get("content","")
                    for lvl in ("easy","medium","hard"):
                        if f"task_level:{lvl}" in c or f"[{lvl}]" in c.lower():
                            return lvl
    return "easy"


def reward_format(completions, prompts=None, **kw):
    """Reward valid JSON action-array format."""
    scores = []
    for i, completion in enumerate(completions):
        text    = _get_text(completion)
        actions = _extract_actions(text)
        if actions is None:
            scores.append(-0.5)
        elif not isinstance(actions, list) or len(actions) == 0:
            scores.append(-0.25)
        else:
            valid = sum(1 for a in actions if isinstance(a,dict) and "action_type" in a)
            scores.append(min(1.0, valid / max(len(actions),1)))
    return scores


def reward_accuracy(completions, prompts=None, **kw):
    """Reward correct verdict."""
    global _PRINTER
    scores = []
    for i, completion in enumerate(completions):
        text    = _get_text(completion)
        actions = _extract_actions(text)
        level   = _task_level_from_prompt(prompts, i)
        if _PRINTER % 10 == 0:
            print(f"[reward_accuracy] sample text[:200]: {text[:200]!r}")
        _PRINTER += 1
        if not actions:
            scores.append(0.0); continue
        grade = _run_episode(actions, task_level=level)
        scores.append(float(grade.get("accuracy", 0.0)))
    return scores


def reward_evidence(completions, prompts=None, **kw):
    """Reward evidence alignment."""
    scores = []
    for i, completion in enumerate(completions):
        text    = _get_text(completion)
        actions = _extract_actions(text)
        level   = _task_level_from_prompt(prompts, i)
        if not actions:
            scores.append(0.0); continue
        grade = _run_episode(actions, task_level=level)
        scores.append(float(grade.get("evidence_alignment", 0.0)))
    return scores


def reward_efficiency(completions, prompts=None, **kw):
    """Reward investigation efficiency."""
    scores = []
    for i, completion in enumerate(completions):
        text    = _get_text(completion)
        actions = _extract_actions(text)
        level   = _task_level_from_prompt(prompts, i)
        if not actions:
            scores.append(0.0); continue
        grade = _run_episode(actions, task_level=level)
        scores.append(float(grade.get("efficiency", 0.0)))
    return scores

print("Reward functions defined.")


## 8. Quick sanity check (reward functions work)

In [ ]:
env_test = SniffTestEnvironment(enable_adversarial=False)
obs_test = env_test.reset(task_level="easy")
sources_str = "\n".join(
    f"- [{s.source_id}] {s.title} ({s.domain}): {s.snippet}"
    for s in obs_test.available_sources
)
print("Claim:", obs_test.claim[:80])

# Minimal valid action list for oracle check
oracle_actions = [
    {"action_type":"search","query":obs_test.claim},
    {"action_type":"open_source","source_id":obs_test.available_sources[0].source_id},
    {"action_type":"submit_verdict","verdict":"false","confidence":0.5,"justification":"test"},
]
grade = _run_episode(oracle_actions, "easy")
print("Oracle grade:", grade)
assert grade, "Reward pipeline broken — check SniffTestEnvironment grader"
print("Sanity check passed.")


## 9. Build dataset

In [ ]:
from datasets import Dataset

CURRICULUM = [
    {"task_level":"easy",   "episodes":300},
    {"task_level":"medium", "episodes":300},
    {"task_level":"hard",   "episodes":200},
]

def make_prompt(task_level: str, claim: str, sources_text: str):
    user_msg = (
        f"[{task_level}] Investigate this claim: {claim}\n\n"
        f"Available sources:\n{sources_text}\n\n"
        "Return ONLY a JSON array of actions (search first, submit_verdict last)."
    )
    return [
        {"role":"system",  "content": SYSTEM_PROMPT},
        {"role":"user",    "content": user_msg},
    ]

def build_dataset(task_level: str, n: int):
    env = SniffTestEnvironment(enable_adversarial=False)
    rows = []
    for _ in range(n):
        obs = env.reset(task_level=task_level)
        sources_text = "\n".join(
            f"- [{s.source_id}] {s.title} ({s.domain}): {s.snippet}"
            for s in obs.available_sources
        )
        rows.append({"prompt": make_prompt(task_level, obs.claim, sources_text)})
    return Dataset.from_list(rows)

print("Dataset builder ready.")


## 10. SFT warm-start (optional)

In [ ]:
# Optional: skip by setting SNIFFTEST_SKIP_SFT=1
import os
if os.getenv("SNIFFTEST_SKIP_SFT","0").lower() not in {"1","true","yes"}:
    import json as _json
    from pathlib import Path
    from datasets import Dataset as _DS
    from trl import SFTConfig, SFTTrainer

    SFT_PATH = REPO_DIR/"data/sft_trajectories.jsonl"
    SFT_OUT  = ARTIFACT_DIR/"sft_warm_start"

    if SFT_PATH.exists():
        rows = []
        for line in SFT_PATH.read_text().splitlines():
            if not line.strip(): continue
            rec = _json.loads(line)
            # Build flat-text example from trajectory
            sources_text = "\n".join(
                f"- [{s['source_id']}] {s['title']} ({s['domain']}): {s['snippet']}"
                for s in rec.get("visible_sources",[])
            )
            actions_json = _json.dumps(rec.get("actions",[]))
            messages = [
                {"role":"system",  "content":SYSTEM_PROMPT},
                {"role":"user",    "content":f"Investigate: {rec['claim']}\n\nSources:\n{sources_text}"},
                {"role":"assistant","content":actions_json},
            ]
            rows.append({"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)})

        sft_ds = _DS.from_list(rows)
        SFT_OUT.mkdir(parents=True, exist_ok=True)

        sft_trainer = SFTTrainer(
            model=model,
            args=SFTConfig(
                output_dir=str(SFT_OUT),
                num_train_epochs=2,
                per_device_train_batch_size=1,
                gradient_accumulation_steps=8,
                learning_rate=2e-4,
                bf16=torch.cuda.is_bf16_supported(),
                fp16=not torch.cuda.is_bf16_supported(),
                logging_steps=10,
                report_to="none",
                dataset_text_field="text",
                max_seq_length=MAX_SEQ_LENGTH,
            ),
            train_dataset=sft_ds,
            processing_class=tokenizer,
        )
        sft_trainer.train()
        sft_trainer.save_model(str(SFT_OUT))
        tokenizer.save_pretrained(str(SFT_OUT))
        print("SFT warm-start saved:", SFT_OUT)
    else:
        print("No SFT trajectories found at", SFT_PATH, "- skipping warm-start")
else:
    print("SNIFFTEST_SKIP_SFT=1: skipping SFT warm-start")


## 11. GRPO curriculum training

This uses Unsloth's patched `GRPOTrainer` with standard `reward_funcs`. The fix from the old notebook: **no `rollout_func`** — rewards are computed inside each reward function directly against `SniffTestEnvironment`.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

USE_BF16 = torch.cuda.is_bf16_supported()
MAX_PROMPT_LEN     = int(os.getenv("SNIFFTEST_MAX_PROMPT_LENGTH","1024"))
MAX_COMPLETION_LEN = int(os.getenv("SNIFFTEST_MAX_COMPLETION_LENGTH","512"))
NUM_GENERATIONS    = int(os.getenv("SNIFFTEST_NUM_GENERATIONS","4"))

all_rewards = []

for stage in CURRICULUM:
    level    = stage["task_level"]
    episodes = stage["episodes"]
    stage_dir = ARTIFACT_DIR / f"{level}_stage"
    stage_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"STAGE: {level.upper()}  ({episodes} episodes)")
    print(f"{'='*60}")

    ds = build_dataset(level, episodes)

    grpo_args = GRPOConfig(
        output_dir                  = str(stage_dir),
        num_train_epochs            = 1,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_generations             = NUM_GENERATIONS,
        learning_rate               = 5e-6,
        warmup_ratio                = 0.1,
        max_prompt_length           = MAX_PROMPT_LEN,
        max_completion_length       = MAX_COMPLETION_LEN,
        bf16                        = USE_BF16,
        fp16                        = not USE_BF16,
        logging_steps               = 5,
        save_steps                  = max(10, episodes//2),
        report_to                   = "none",
        gradient_checkpointing      = True,
    )

    trainer = GRPOTrainer(
        model            = model,
        processing_class = tokenizer,
        reward_funcs     = [reward_format, reward_accuracy, reward_evidence, reward_efficiency],
        args             = grpo_args,
        train_dataset    = ds,
    )

    # Confirm trainer class (Unsloth patches it — that's fine here, reward_funcs is respected)
    print(f"  Trainer: {type(trainer).__module__}.{type(trainer).__name__}")
    trainer.train()

    # Collect reward history
    for entry in trainer.state.log_history:
        r = entry.get("reward") or entry.get("train/reward")
        if r is not None:
            all_rewards.append({"level": level, "reward": float(r)})

    model.save_pretrained(str(stage_dir))
    tokenizer.save_pretrained(str(stage_dir))
    print(f"  Stage checkpoint: {stage_dir}")

print("\nTraining complete.")


## 12. Save final model

In [ ]:
FINAL_DIR = ARTIFACT_DIR / "final_merged"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

if hasattr(model, "save_pretrained_merged"):
    model.save_pretrained_merged(str(FINAL_DIR), tokenizer, save_method="merged_16bit")
else:
    model.save_pretrained(str(FINAL_DIR))
    tokenizer.save_pretrained(str(FINAL_DIR))

(ARTIFACT_DIR/"reward_history.json").write_text(json.dumps(all_rewards, indent=2))
print("Final model:", FINAL_DIR)
print("Reward history entries:", len(all_rewards))


## 13. Training curve

In [ ]:
import matplotlib.pyplot as plt, json as _json
history = _json.loads((ARTIFACT_DIR/"reward_history.json").read_text())
colors = {"easy":"#22c55e","medium":"#f59e0b","hard":"#ef4444"}
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax, level in zip(axes,["easy","medium","hard"]):
    rs = [e["reward"] for e in history if e["level"]==level]
    if not rs: ax.set_title(level); continue
    w = 20
    rolling = [sum(rs[max(0,i-w):i+1])/min(i+1,w) for i in range(len(rs))]
    ax.plot(rs, alpha=0.2, color=colors[level])
    ax.plot(rolling, color=colors[level], lw=2, label="rolling avg")
    ax.set_title(level.capitalize(), fontweight="bold")
    ax.set_xlabel("Step"); ax.set_ylabel("Reward")
    ax.legend()
plt.suptitle("SniffTest GRPO Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()
